# Reference-image conditioning path in OmniGen2 (evidence)

Confirms, from source, how reference images reach the model during subject-driven generation, and reconciles it with the paper's Figure 2.

Tested claim: in the generation pipeline this thesis uses (`pipeline_omnigen2.py` / `OmniGen2Pipeline`), the Qwen2.5-VL MLLM is called on **text token ids only**; reference images enter through the **VAE** into the diffusion transformer. The ViT to text-transformer path in Figure 2 is the **chat/understanding** pipeline (`pipeline_omnigen2_chat.py`), which this thesis does not use.

Each cell prints the exact source lines and asserts the claim. The final cell (optional) loads the model on an A100 and logs the MLLM call arguments at runtime.

In [1]:
# resolve the omnigen2 source this environment runs; fall back to a shallow clone for files outside the package
import importlib.util, os, subprocess

def ensure_clone():
    if not os.path.exists("OmniGen2"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/VectorSpaceLab/OmniGen2.git"], check=True)
    return "OmniGen2"

spec = importlib.util.find_spec("omnigen2")   # locates without executing the package
if spec and spec.submodule_search_locations and \
   os.path.exists(os.path.join(list(spec.submodule_search_locations)[0],
                               "pipelines/omnigen2/pipeline_omnigen2.py")):
    PKG, SRC = list(spec.submodule_search_locations)[0], "installed omnigen2 package"
else:
    PKG, SRC = f"{ensure_clone()}/omnigen2", "shallow clone (github HEAD)"

GEN   = f"{PKG}/pipelines/omnigen2/pipeline_omnigen2.py"
CHAT  = f"{PKG}/pipelines/omnigen2/pipeline_omnigen2_chat.py"
TRAN  = f"{PKG}/models/transformers/transformer_omnigen2.py"
INFER = f"{ensure_clone()}/omnicontext/inference.py"   # omnicontext is not part of the pip package

print("reading from:", SRC)
print("package dir :", PKG)
for f in [GEN, CHAT, TRAN, INFER]:
    print("   [ok]  " + f if os.path.exists(f) else "   [MISSING]  " + f)

reading from: shallow clone (github HEAD)
package dir : OmniGen2/omnigen2
   [ok]  OmniGen2/omnigen2/pipelines/omnigen2/pipeline_omnigen2.py
   [ok]  OmniGen2/omnigen2/pipelines/omnigen2/pipeline_omnigen2_chat.py
   [ok]  OmniGen2/omnigen2/models/transformers/transformer_omnigen2.py
   [ok]  OmniGen2/omnicontext/inference.py


In [2]:
import ast

def func_src(path, name, cls=None):
    src = open(path, encoding="utf-8").read(); lines = src.splitlines(); found = []
    def walk(node, inside):
        for n in ast.iter_child_nodes(node):
            if isinstance(n, ast.ClassDef):
                walk(n, n.name)
            elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == name \
                 and (cls is None or inside == cls):
                found.append((n.lineno, n.end_lineno))
    walk(ast.parse(src), None)
    if not found: raise LookupError(name + " not in " + path)
    s, e = found[0]
    return s, "\n".join(f"{i:>4} | {lines[i-1]}" for i in range(s, e + 1))

def grep_line(path, needle):
    for i, l in enumerate(open(path, encoding="utf-8").read().splitlines(), 1):
        if needle in l: return i, l
    raise LookupError(repr(needle) + " not in " + path)

def grep_all(path, needle):
    hits = [(i, l) for i, l in enumerate(open(path, encoding="utf-8").read().splitlines(), 1) if needle in l]
    if not hits: raise LookupError(repr(needle) + " not in " + path)
    return hits

def show(title, body):
    print("\n===== " + title + " =====")
    print(body)

## Claim 1 - generation pipeline: the MLLM is fed text only

`_get_qwen2_prompt_embeds` in `pipeline_omnigen2.py` tokenizes the prompt (text) and calls the MLLM on token ids. No image tensor is passed.

In [3]:
s, body = func_src(GEN, "_get_qwen2_prompt_embeds")
show(f"pipeline_omnigen2.py :: _get_qwen2_prompt_embeds  (line {s})", body)

assert "self.processor.tokenizer(" in body, "expected the TOKENIZER (text) call"
assert "self.mllm(" in body and "text_input_ids" in body, "expected mllm(text_input_ids, ...)"
assert "images=" not in body, "no image tensor should be passed to the MLLM here"
print("\n[PASS] generation pipeline: MLLM receives text token ids only (no images).")


===== pipeline_omnigen2.py :: _get_qwen2_prompt_embeds  (line 274) =====
 274 |     def _get_qwen2_prompt_embeds(
 275 |         self,
 276 |         prompt: Union[str, List[str]],
 277 |         device: Optional[torch.device] = None,
 278 |         max_sequence_length: int = 256,
 279 |     ) -> Tuple[torch.Tensor, torch.Tensor]:
 280 |         """
 281 |         Get prompt embeddings from the Qwen2 text encoder.
 282 | 
 283 |         Args:
 284 |             prompt: The prompt or list of prompts to encode.
 285 |             device: The device to place the embeddings on. If None, uses the pipeline's device.
 286 |             max_sequence_length: Maximum sequence length for tokenization.
 287 | 
 288 |         Returns:
 289 |             Tuple[torch.Tensor, torch.Tensor]: A tuple containing:
 290 |                 - The prompt embeddings tensor
 291 |                 - The attention mask tensor
 292 | 
 293 |         Raises:
 294 |             Warning: If the input text is truncate

## Contrast - chat pipeline: the MLLM is fed images (the Figure 2 path)

The `_get_qwen2_prompt_embeds` in `pipeline_omnigen2_chat.py` calls the full processor with `images=`, which produces pixel features, then passes `**inputs` to the MLLM. This is the ViT to text-transformer path drawn in Figure 2. The thesis does not use this pipeline.

In [4]:
s, body = func_src(CHAT, "_get_qwen2_prompt_embeds")
show(f"pipeline_omnigen2_chat.py :: _get_qwen2_prompt_embeds  (line {s})", body)

assert "self.processor(" in body and "images=" in body, "chat: expected processor(images=...)"
assert "self.mllm(" in body, "chat: expected mllm(**inputs)"
print("\n[PASS] chat pipeline: processor(images=...) produces pixel features -> mllm(**inputs).")
print("       That is the Figure 2 ViT path; the generation pipeline above does not do this.")


===== pipeline_omnigen2_chat.py :: _get_qwen2_prompt_embeds  (line 279) =====
 279 |     def _get_qwen2_prompt_embeds(
 280 |         self, 
 281 |         prompt: Union[str, List[str]],
 282 |         input_images = None, 
 283 |         device: Optional[torch.device] = None,
 284 |         use_only_text_hidden_states: bool = True,
 285 |         ) -> Tuple[torch.Tensor, torch.Tensor]:
 286 |         """
 287 |         Get prompt embeddings from the Qwen2 text encoder.
 288 | 
 289 |         Args:
 290 |             prompt: The prompt or list of prompts to encode.
 291 |             device: The device to place the embeddings on. If None, uses the pipeline's device.
 292 | 
 293 |         Returns:
 294 |             Tuple[torch.Tensor, torch.Tensor]: A tuple containing:
 295 |                 - The prompt embeddings tensor
 296 |                 - The attention mask tensor
 297 | 
 298 |         Raises:
 299 |             Warning: If the input text is truncated due to sequence length 

## Claim 2 - reference images enter via the VAE, into the diffusion transformer

`prepare_image` preprocesses each reference and runs it through `encode_vae` (VAE). The resulting latents are handed to the transformer as `ref_image_hidden_states`.

In [5]:
s1, b1 = func_src(GEN, "encode_vae")
s2, b2 = func_src(GEN, "prepare_image")
show(f"encode_vae  (line {s1})", b1)
show(f"prepare_image  (line {s2})", b2)

# every transformer call site that receives references (all CFG branches, not just the first)
hits = grep_all(GEN, "ref_image_hidden_states=")
print("\ntransformer call sites receiving references:")
for ln, txt in hits:
    print(f"{ln:>4} | {txt.strip()}")

srcs = {t.split("=", 1)[1].strip().rstrip(",") for _, t in hits}
print("\nvalues passed:", sorted(srcs))
assert srcs <= {"ref_latents", "None"}, f"unexpected reference source: {srcs}"
assert "self.vae.encode(" in open(GEN, encoding="utf-8").read(), "expected vae.encode"
print("\n[PASS] references: image_processor.preprocess -> VAE.encode -> ref_image_hidden_states.")
print("       Every branch gets the VAE latents or None; nothing else reaches the transformer.")


===== encode_vae  (line 217) =====
 217 |     def encode_vae(self, img: torch.FloatTensor) -> torch.FloatTensor:
 218 |         """
 219 |         Encode an image into the VAE latent space.
 220 | 
 221 |         Args:
 222 |             img: The input image tensor to encode.
 223 | 
 224 |         Returns:
 225 |             torch.FloatTensor: The encoded latent representation.
 226 |         """
 227 |         z0 = self.vae.encode(img.to(dtype=self.vae.dtype)).latent_dist.sample()
 228 |         if self.vae.config.shift_factor is not None:
 229 |             z0 = z0 - self.vae.config.shift_factor
 230 |         if self.vae.config.scaling_factor is not None:
 231 |             z0 = z0 * self.vae.config.scaling_factor
 232 |         z0 = z0.to(dtype=self.vae.dtype)
 233 |         return z0

===== prepare_image  (line 235) =====
 235 |     def prepare_image(
 236 |         self,
 237 |         images: Union[List[PIL.Image.Image], PIL.Image.Image],
 238 |         batch_size: int,
 239 |

## Claim 2b - which CFG branches see the references
The four call sites above are the dual-CFG branches. negative_prompt feeds two of the three; the "uncond" branch is the negative prompt with references dropped.

In [6]:
# dual-CFG: how the three branches combine (the training-free levers)
gen_src = open(GEN, encoding="utf-8").read()
i = gen_src.find("model_pred = model_pred_uncond")
print("----- guidance combination -----")
print(gen_src[i:i + 220])
print("""
branches (matching the call sites above):
  model_pred        = predict(prompt_embeds,          ref_latents)   full: instruction + refs
  model_pred_ref    = predict(negative_prompt_embeds, ref_latents)   refs, negative text
  model_pred_uncond = predict(negative_prompt_embeds, None)          negative text, no refs

  image_guidance_scale * (ref  - uncond)  -> toward the reference subject   (SC)
  text_guidance_scale  * (full - ref)     -> toward the instruction         (PF)

negative_prompt feeds TWO of the three branches; the "uncond" branch is not
unconditional -- it is the negative prompt with references dropped.""")

----- guidance combination -----
model_pred = model_pred_uncond + image_guidance_scale * (model_pred_ref - model_pred_uncond) + \
                        text_guidance_scale * (model_pred - model_pred_ref)
                elif text_guidance_scale > 1.0:

branches (matching the call sites above):
  model_pred        = predict(prompt_embeds,          ref_latents)   full: instruction + refs
  model_pred_ref    = predict(negative_prompt_embeds, ref_latents)   refs, negative text
  model_pred_uncond = predict(negative_prompt_embeds, None)          negative text, no refs

  image_guidance_scale * (ref  - uncond)  -> toward the reference subject   (SC)
  text_guidance_scale  * (full - ref)     -> toward the instruction         (PF)

negative_prompt feeds TWO of the three branches; the "uncond" branch is not
unconditional -- it is the negative prompt with references dropped.


## Claim 3 - references are indexed by a learned embedding in the transformer, not the MLLM

The diffusion transformer holds a learned `image_index_embedding` (max 5) and adds it to each reference's hidden states. The order/identity of references is handled on the diffusion side.

In [7]:
ln, txt = grep_line(TRAN, "self.image_index_embedding = nn.Parameter")
print(f"transformer_omnigen2.py (line {ln}):\n{ln:>4} | {txt.strip()}")
la, ta = grep_line(TRAN, "self.image_index_embedding[")
print(f"\napplied per reference (line {la}):\n{la:>4} | {ta.strip()}")
assert "nn.Parameter" in txt, "expected a learned parameter"
print("\n[PASS] references are distinguished by a diffusion-side learned index embedding, not by the MLLM.")

transformer_omnigen2.py (line 381):
 381 | self.image_index_embedding = nn.Parameter(torch.randn(5, hidden_size)) # support max 5 ref images

applied per reference (line 435):
 435 | ref_image_hidden_states[i, shift:shift + ref_img_len, :] = ref_image_hidden_states[i, shift:shift + ref_img_len, :] + self.image_index_embedding[j]

[PASS] references are distinguished by a diffusion-side learned index embedding, not by the MLLM.


## Which pipeline the thesis uses

OmniContext generation loads `OmniGen2Pipeline` (= `pipeline_omnigen2.py`, the text-only-MLLM path). The thesis generate notebook imports the same class.

In [8]:
ln, txt = grep_line(INFER, "import OmniGen2Pipeline")
print(f"omnicontext/inference.py (line {ln}):\n{ln:>4} | {txt.strip()}")
assert "pipeline_omnigen2 import" in txt and "chat" not in txt, "expected the generation pipeline import"
print("\n[PASS] OmniContext (and the thesis generate notebook) load the generation pipeline shown in Claim 1.")

omnicontext/inference.py (line 19):
  19 | from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline

[PASS] OmniContext (and the thesis generate notebook) load the generation pipeline shown in Claim 1.


## Optional: runtime confirmation on the actual model (A100 + weights)

Loads OmniGen2, hooks the MLLM forward, runs one generation with a reference image, and prints the arguments the MLLM actually received. Set `RUN_RUNTIME = True` to execute. Static evidence above already settles the claim; this is the empirical clincher.

In [9]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")
sys.path.insert(0, REPO)
os.chdir(REPO)

OmniGen2 already present.


In [10]:
RUN_RUNTIME = True
MODEL = "OmniGen2/OmniGen2"

if RUN_RUNTIME:
    import torch
    from PIL import Image
    from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
    from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

    dtype = torch.bfloat16
    pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
    pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
    pipe = pipe.to("cuda")

    # Qwen2.5-VL pixel_values are 2-D [num_patches, patch_dim], so a rank test cannot detect them.
    # Test by kwarg name and by dtype: token ids are integer, pixel features are float.
    IMAGE_KWARGS = ("pixel_values", "image_grid_thw", "pixel_values_videos", "video_grid_thw")
    calls = []                                   # every MLLM call, not just the last
    def mllm_pre_hook(mod, args, kwargs):
        tensors = [a for a in args if torch.is_tensor(a)]
        calls.append({
            "arg_shapes":   [tuple(a.shape) for a in tensors],
            "arg_dtypes":   [str(a.dtype) for a in tensors],
            "kwarg_keys":   sorted(kwargs.keys()),
            "image_kwargs": [k for k in kwargs if k in IMAGE_KWARGS and kwargs[k] is not None],
            "float_kwargs": [k for k, v in kwargs.items() if torch.is_tensor(v) and v.is_floating_point()],
        })
    h = pipe.mllm.register_forward_pre_hook(mllm_pre_hook, with_kwargs=True)

    vae_calls = {"n": 0}
    _orig_encode = pipe.vae.encode
    def counting_encode(*a, **k):
        vae_calls["n"] += 1
        return _orig_encode(*a, **k)
    pipe.vae.encode = counting_encode

    ref = Image.new("RGB", (512, 512), (127, 127, 127))   # content irrelevant to the call signature
    _ = pipe(prompt="the object is <img><|image_1|></img>", input_images=[ref],
             width=512, height=512, num_inference_steps=1, align_res=False,
             generator=torch.Generator("cuda").manual_seed(0))
    h.remove()
    pipe.vae.encode = _orig_encode

    print(f"MLLM forward calls: {len(calls)}   (encode_prompt runs it twice: positive + negative prompt)")
    for n, c in enumerate(calls, 1):
        print(f"  call {n}: positional {c['arg_shapes']} {c['arg_dtypes']} | kwargs {c['kwarg_keys']}")
        print(f"          image kwargs: {c['image_kwargs'] or 'none'} | float tensors: {c['float_kwargs'] or 'none'}")
    print("VAE.encode call count:", vae_calls["n"])

    assert calls, "hook never fired"
    for c in calls:
        assert not c["image_kwargs"], f"MLLM received image kwargs: {c['image_kwargs']}"
        assert not c["float_kwargs"], f"MLLM received float (pixel-like) tensors: {c['float_kwargs']}"
        assert all(d.startswith(("torch.int", "torch.long")) for d in c["arg_dtypes"]), f"not token ids: {c['arg_dtypes']}"
        assert all(len(s) == 2 for s in c["arg_shapes"]), f"expected [batch, seq_len], got {c['arg_shapes']}"
    assert vae_calls["n"] >= 1, "reference did not go through the VAE"
    print("\n[PASS] runtime: every MLLM call received 2-D integer token ids only; the reference went through the VAE.")
else:
    print("RUN_RUNTIME=False -- skipping model load (static evidence above is sufficient).")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

Expected types for transformer: (<class 'omnigen2.models.transformers.transformer_omnigen2.OmniGen2Transformer2DModel'>,), got <class 'diffusers_modules.local.transformer_omnigen2.OmniGen2Transformer2DModel'>.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

MLLM forward calls: 2   (encode_prompt runs it twice: positive + negative prompt)
  call 1: positional [(1, 39)] ['torch.int64'] | kwargs ['attention_mask', 'output_hidden_states']
          image kwargs: none | float tensors: none
  call 2: positional [(1, 25)] ['torch.int64'] | kwargs ['attention_mask', 'output_hidden_states']
          image kwargs: none | float tensors: none
VAE.encode call count: 1

[PASS] runtime: every MLLM call received 2-D integer token ids only; the reference went through the VAE.


## Conclusion

Confirmed:

- Generation path (`pipeline_omnigen2.py`, loaded by OmniContext and the thesis generate notebook): `self.processor.tokenizer(prompt)` then `self.mllm(text_input_ids, ...)`. No image tensor reaches the MLLM.
- Reference images: `image_processor.preprocess` then `VAE.encode` then `ref_image_hidden_states`, distinguished by a learned `image_index_embedding` (max 5) in the diffusion transformer.
- The Figure 2 ViT to MLLM path lives in `pipeline_omnigen2_chat.py` (`processor(images=...)` then `mllm(**inputs)`), which the thesis does not use.

Thesis wording: "In OmniGen2's generation pipeline, reference images are VAE-encoded into the diffusion transformer as `ref_image_hidden_states` (with a learned per-image index embedding); the Qwen2.5-VL MLLM receives text tokens only. The ViT to text-transformer path in Figure 2 is the understanding/chat pipeline, not the reference-conditioning path used here."

Not: "the model cannot encode images." It can, via the chat pipeline. The point is that the generation pipeline does not route references through it.